<a href="https://colab.research.google.com/github/ms-starryvoid/ML_Lab_DataSet/blob/main/Programs/ID3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import math
from collections import Counter
import pandas as pd

# Function to calculate entropy
def entropy(data, target_attr):
    values = [record[target_attr] for record in data]
    counter = Counter(values)
    total = len(values)
    return -sum((count/total) * math.log2(count/total) for count in counter.values())

# Function to calculate information gain
def information_gain(data, attr, target_attr):
    total_entropy = entropy(data, target_attr)
    values = set(record[attr] for record in data)
    subset_entropy = 0.0

    for value in values:
        subset = [record for record in data if record[attr] == value]
        subset_entropy += (len(subset)/len(data)) * entropy(subset, target_attr)

    return total_entropy - subset_entropy

# Function to build the tree using ID3
def id3(data, attributes, target_attr):
    values = [record[target_attr] for record in data]

    # If all examples have the same classification, return it
    if values.count(values[0]) == len(values):
        return values[0]

    # If no attributes left, return majority value
    if not attributes:
        return Counter(values).most_common(1)[0][0]

    # Choose attribute with highest information gain
    gains = [(attr, information_gain(data, attr, target_attr)) for attr in attributes]
    best_attr = max(gains, key=lambda x: x[1])[0]
    print("\nbest attribute :",best_attr)
    tree = {best_attr: {}}

    for value in set(record[best_attr] for record in data):
        subset = [record for record in data if record[best_attr] == value]
        if not subset:
            tree[best_attr][value] = Counter(values).most_common(1)[0][0]
        else:
            new_attrs = [a for a in attributes if a != best_attr]
            tree[best_attr][value] = id3(subset, new_attrs, target_attr)

    return tree

# Pretty-print the tree
def print_tree(tree, depth=0):
    if isinstance(tree, dict):
        for attr, branches in tree.items():
            for value, subtree in branches.items():
                print("|   " * depth + f"{attr} = {value}:")
                print_tree(subtree, depth + 1)
    else:
        print("|   " * depth + f"-> {tree}")

# Function to predict outcome for a new sample
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attr = next(iter(tree))
    value = sample.get(attr)

    if value not in tree[attr]:
        return None  # Unknown case

    return predict(tree[attr][value], sample)


csv_path = "https://raw.githubusercontent.com/ms-starryvoid/ML_Lab_DataSet/refs/heads/main/Lab_data/PlayTennis.csv"
df = pd.read_csv(csv_path)
print("dataset\n",df)
data = df.to_dict(orient='records')

attributes = list(df.columns)
target_attr = attributes.pop(-1)  # Assuming last column is the target
# Now attributes has all columns except the target

# Build decision tree
tree = id3(data, attributes, target_attr)

# Print the tree
print("Decision Tree:")
print_tree(tree)

# Example prediction (modify as needed based on your dataset's columns)
sample_day = {
    "outlook": "overcast",
    "temp": "hot",
    "humidity": "high",
    "windy": "False"
} # Just an example using first row features
print("\nPrediction for sample day:", predict(tree, sample_day))


dataset
      outlook  temp humidity  windy play
0      sunny   hot     high  False   no
1      sunny   hot     high   True   no
2   overcast   hot     high  False  yes
3      rainy  mild     high  False  yes
4      rainy  cool   normal  False  yes
5      rainy  cool   normal   True   no
6   overcast  cool   normal   True  yes
7      sunny  mild     high  False   no
8      sunny  cool   normal  False  yes
9      rainy  mild   normal  False  yes
10     sunny  mild   normal   True  yes
11  overcast  mild     high   True  yes
12  overcast   hot   normal  False  yes
13     rainy  mild     high   True   no

best attribute : outlook

best attribute : windy

best attribute : humidity
Decision Tree:
outlook = overcast:
|   -> yes
outlook = rainy:
|   windy = False:
|   |   -> yes
|   windy = True:
|   |   -> no
outlook = sunny:
|   humidity = normal:
|   |   -> yes
|   humidity = high:
|   |   -> no

Prediction for sample day: yes
